## Exercise 1: Set up the Solaris Energy Data Platform

Before you can ingest any data, you need to establish the Unity Catalog structure: a **catalog** to represent the platform, **schemas** to organise data by processing layer, and a **volume** to store raw files.

After completing this exercise, return to the lab setup page to verify the objects you created in **Catalog Explorer**.

### Task 1: Set up the Unity Catalog structure

Since you've already practiced creating catalogs, schemas, and volumes in a previous lab, this setup code is provided for you — simply run the cell below to prepare your environment.

The code creates the following Unity Catalog objects:
- A **catalog** named `solaris_lab` — the top-level namespace for the Solaris Energy platform
- A **bronze schema** inside `solaris_lab` — for raw ingested data, exactly as it arrives from source systems
- A **silver schema** inside `solaris_lab` — for cleansed and enriched data
- A **managed volume** named `raw_files` inside `solaris_lab.bronze` — the landing zone for incoming CSV files from field sensors and SCADA systems

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS solaris_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Solaris Energy analytics platform — solar farms and wind turbines across Europe'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS solaris_lab.bronze
  COMMENT 'Raw ingested data — unmodified, as received from field systems';

CREATE SCHEMA IF NOT EXISTS solaris_lab.silver
  COMMENT 'Cleansed and enriched data — ready for analytics and reporting';
  
CREATE VOLUME IF NOT EXISTS solaris_lab.bronze.raw_files
  COMMENT 'Landing zone for raw CSV files from SCADA systems and field sensors';

### ✅ Provided: Populate the volume with sample data

Run the cell below to write two sample CSV files into your volume:
- `solar_readings.csv` — energy output readings from solar panels at sites in Spain and Germany
- `turbine_events.csv` — maintenance and alert events from wind turbines in the Netherlands and Denmark

You will use these files in Exercises 2, 3, and 4.

In [ ]:
# ✅ Run this cell — it writes sample data files to your volume

solar_csv = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R001,SITE_ES_001,P-ES-001,2025-06-01 06:30:00,8.2,210,18.4,Normal
R002,SITE_ES_001,P-ES-002,2025-06-01 06:30:00,7.9,205,18.6,Normal
R003,SITE_ES_001,P-ES-003,2025-06-01 06:30:00,8.5,215,18.2,Normal
R004,SITE_ES_001,P-ES-004,2025-06-01 06:30:00,0.0,205,18.5,Fault
R005,SITE_ES_001,P-ES-005,2025-06-01 06:30:00,7.6,200,18.8,Normal
R006,SITE_ES_001,P-ES-001,2025-06-01 09:00:00,142.3,750,28.1,Normal
R007,SITE_ES_001,P-ES-002,2025-06-01 09:00:00,138.7,745,28.4,Normal
R008,SITE_ES_001,P-ES-003,2025-06-01 09:00:00,145.1,755,27.9,Normal
R009,SITE_ES_001,P-ES-004,2025-06-01 09:00:00,0.0,750,28.2,Fault
R010,SITE_ES_001,P-ES-005,2025-06-01 09:00:00,136.4,740,28.7,Normal
R011,SITE_ES_001,P-ES-001,2025-06-01 12:00:00,228.5,1150,41.3,Normal
R012,SITE_ES_001,P-ES-002,2025-06-01 12:00:00,221.4,1140,41.8,Warning
R013,SITE_ES_001,P-ES-003,2025-06-01 12:00:00,235.8,1160,40.9,Normal
R014,SITE_ES_001,P-ES-004,2025-06-01 12:00:00,0.0,1150,41.5,Fault
R015,SITE_ES_001,P-ES-005,2025-06-01 12:00:00,219.6,1135,42.1,Warning
R016,SITE_DE_001,P-DE-001,2025-06-01 06:30:00,4.1,120,12.3,Normal
R017,SITE_DE_001,P-DE-002,2025-06-01 06:30:00,4.3,125,12.1,Normal
R018,SITE_DE_001,P-DE-003,2025-06-01 06:30:00,3.8,118,12.5,Normal
R019,SITE_DE_001,P-DE-004,2025-06-01 06:30:00,4.0,122,12.4,Normal
R020,SITE_DE_001,P-DE-005,2025-06-01 06:30:00,4.2,124,12.2,Normal
R021,SITE_DE_001,P-DE-001,2025-06-01 09:00:00,98.7,540,19.8,Normal
R022,SITE_DE_001,P-DE-002,2025-06-01 09:00:00,101.2,548,19.6,Normal
R023,SITE_DE_001,P-DE-003,2025-06-01 09:00:00,96.4,535,20.1,Normal
R024,SITE_DE_001,P-DE-004,2025-06-01 09:00:00,99.8,542,19.9,Normal
R025,SITE_DE_001,P-DE-005,2025-06-01 09:00:00,102.1,550,19.4,Normal
R026,SITE_DE_001,P-DE-001,2025-06-01 12:00:00,187.3,980,31.2,Normal
R027,SITE_DE_001,P-DE-002,2025-06-01 12:00:00,191.6,992,30.8,Normal
R028,SITE_DE_001,P-DE-003,2025-06-01 12:00:00,184.9,975,31.5,Normal
R029,SITE_DE_001,P-DE-004,2025-06-01 12:00:00,188.7,984,31.0,Normal
R030,SITE_DE_001,P-DE-005,2025-06-01 12:00:00,193.2,996,30.6,Normal
R031,SITE_ES_001,P-ES-001,2025-06-02 12:00:00,232.1,1165,42.7,Normal
R032,SITE_ES_001,P-ES-002,2025-06-02 12:00:00,226.9,1155,43.1,Normal
R033,SITE_ES_001,P-ES-003,2025-06-02 12:00:00,239.4,1175,42.3,Normal
R034,SITE_ES_001,P-ES-004,2025-06-02 12:00:00,225.2,1148,43.4,Normal
R035,SITE_ES_001,P-ES-005,2025-06-02 12:00:00,231.7,1162,42.9,Normal
"""

turbine_csv = """event_id,turbine_id,site_id,event_timestamp,event_type,severity,description,resolved
EVT001,T-NL-001,WIND_NL_001,2025-06-01 02:14:00,Alert,Medium,Vibration level 18% above baseline threshold,true
EVT002,T-NL-002,WIND_NL_001,2025-06-01 03:47:00,Fault,High,Gearbox oil temperature exceeded 85 degrees C,true
EVT003,T-NL-003,WIND_NL_001,2025-06-01 07:30:00,Inspection,Low,Scheduled 6-month rotor blade inspection,true
EVT004,T-NL-004,WIND_NL_001,2025-06-01 09:12:00,Alert,Low,Wind speed sensor calibration drift detected,false
EVT005,T-NL-005,WIND_NL_001,2025-06-01 11:55:00,Maintenance,Medium,Yaw motor lubrication service,true
EVT006,T-DK-001,WIND_DK_001,2025-06-01 01:05:00,Alert,Critical,Blade pitch control failure - emergency stop activated,true
EVT007,T-DK-002,WIND_DK_001,2025-06-01 04:22:00,Fault,Medium,Power converter overload - output capped at 60%,false
EVT008,T-DK-003,WIND_DK_001,2025-06-01 06:45:00,Inspection,Low,Annual safety and compliance inspection,true
EVT009,T-DK-004,WIND_DK_001,2025-06-01 10:33:00,Maintenance,Low,Tower bolt torque check completed,true
EVT010,T-DK-005,WIND_DK_001,2025-06-01 14:18:00,Alert,Medium,Ice accretion on blades - de-icing initiated,true
EVT011,T-NL-001,WIND_NL_001,2025-06-02 05:30:00,Maintenance,Low,Hydraulic fluid top-up completed,true
EVT012,T-NL-002,WIND_NL_001,2025-06-02 08:00:00,Alert,High,Generator winding temperature 12 degrees C above limit,false
EVT013,T-NL-003,WIND_NL_001,2025-06-02 09:45:00,Fault,Medium,Anemometer replaced after sensor malfunction,true
EVT014,T-NL-004,WIND_NL_001,2025-06-02 12:10:00,Inspection,Low,Control system firmware update applied,true
EVT015,T-NL-005,WIND_NL_001,2025-06-02 15:22:00,Alert,Low,Brake pad wear indicator triggered,false
EVT016,T-DK-001,WIND_DK_001,2025-06-02 03:18:00,Maintenance,Medium,Main bearing grease replenishment,true
EVT017,T-DK-002,WIND_DK_001,2025-06-02 07:55:00,Alert,High,Pitch battery backup low - replacement scheduled,false
EVT018,T-DK-003,WIND_DK_001,2025-06-02 10:40:00,Fault,Critical,Transformer fault - turbine offline for 4 hours,true
EVT019,T-DK-004,WIND_DK_001,2025-06-02 13:15:00,Inspection,Low,Cooling system filters cleaned,true
EVT020,T-DK-005,WIND_DK_001,2025-06-02 16:50:00,Maintenance,Medium,Slip ring inspection and cleaning completed,true
"""

with open("/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv", "w") as f:
    f.write(solar_csv.strip())

with open("/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv", "w") as f:
    f.write(turbine_csv.strip())

print("✅ Sample data written to /Volumes/solaris_lab/bronze/raw_files/")

## Exercise 2: Batch Ingestion with PySpark DataFrames

The SCADA system at Solaris Energy exports daily solar panel readings as CSV files. Your first ingestion task is to read the latest file from the volume, inspect its contents, and write it as a managed Delta table in the `bronze` layer.

After completing this exercise, return to the lab setup page to inspect the table in **Catalog Explorer**.

### Task 2.1 — Read the solar readings CSV into a DataFrame

Use the `spark.read` API to load `solar_readings.csv` from the volume. The file has a header row and you should let Spark infer the column data types automatically.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I read a CSV file from a Unity Catalog volume into a PySpark DataFrame with header and schema inference enabled?"*

**Hints:**
- The volume path is `/Volumes/solaris_lab/bronze/raw_files/solar_readings.csv`
- Use `.format("csv")`, `.option("header", "true")`, `.option("inferSchema", "true")`, and `.load(...)`

In [ ]:
# TODO: Read solar_readings.csv from the volume into a DataFrame named df_solar

df_solar = ...

### Task 2.2 — Inspect the schema and sample rows

Before writing data to a table, it is good practice to verify the inferred schema and check a sample of records. Confirm that `power_kw`, `irradiance_w_m2`, and `temperature_c` are numeric types.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I print the schema of a PySpark DataFrame and display the first few rows?"*

**Hints:**
- Use `df_solar.printSchema()` to view column names and data types
- Use `display(df_solar)` to preview the data

In [ ]:
# TODO: Print the schema of df_solar


# TODO: Display a sample of the data

### Task 2.3 — Write the DataFrame to a Delta table

Persist the solar readings as a managed Delta table at `solaris_lab.bronze.solar_readings`. Azure Databricks uses Delta Lake format by default, which provides ACID transactions and time travel.

Use `overwrite` mode so the table is fully replaced each time the cell runs (appropriate for an initial full load).

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I write a PySpark DataFrame to a Unity Catalog Delta table using saveAsTable in overwrite mode?"*

**Hints:**
- Use `df_solar.write.mode("overwrite").saveAsTable("solaris_lab.bronze.solar_readings")`

In [ ]:
# TODO: Write df_solar to solaris_lab.bronze.solar_readings in overwrite mode

### Task 2.4 — Verify the ingested table

Query the Delta table you just created to confirm all rows were loaded. Also filter for rows where `status = 'Fault'` to check that fault readings from panel `P-ES-004` are present.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I count rows in a Databricks SQL table and filter by a column value?"*

**Hints:**
- Use `SELECT COUNT(*) FROM solaris_lab.bronze.solar_readings`
- Add a `WHERE status = 'Fault'` clause to filter faults

In [ ]:
%sql
-- TODO: Count all rows in solaris_lab.bronze.solar_readings


-- TODO: Count only rows where status = 'Fault'

## Exercise 3: SQL-Based Ingestion

The operations team delivers wind turbine event logs as CSV files that accumulate in the volume over time. Rather than reloading the entire file each run, you will use `COPY INTO` for idempotent, incremental loading. You will then build a silver-layer summary table using `CREATE TABLE AS SELECT`.

### Task 3.1 — Create the turbine events target table

`COPY INTO` requires the destination table to exist before you run it. Create a Delta table `solaris_lab.bronze.turbine_events` with the following columns:

| Column | Type |
|---|---|
| `event_id` | `STRING` |
| `turbine_id` | `STRING` |
| `site_id` | `STRING` |
| `event_timestamp` | `TIMESTAMP` |
| `event_type` | `STRING` |
| `severity` | `STRING` |
| `description` | `STRING` |
| `resolved` | `BOOLEAN` |

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a Delta table in Databricks SQL with explicit column types, using CREATE TABLE IF NOT EXISTS?"*

**Hints:**
- Use `CREATE TABLE IF NOT EXISTS solaris_lab.bronze.turbine_events (...)`
- Use `USING DELTA` (the recommended storage format in Azure Databricks)

In [ ]:
%sql
-- TODO: Create the Delta table solaris_lab.bronze.turbine_events
-- Include all 8 columns listed above with correct data types

### Task 3.2 — Load turbine events with COPY INTO

Use `COPY INTO` to load `turbine_events.csv` from the volume. `COPY INTO` is idempotent — it tracks which files have already been loaded, so re-running the command will not create duplicate rows.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I use COPY INTO in Databricks SQL to load a CSV file from a Unity Catalog volume, including FORMAT_OPTIONS for header and type inference?"*

**Hints:**
- Point the `FROM` clause to `/Volumes/solaris_lab/bronze/raw_files/turbine_events.csv`
- Set `FILEFORMAT = CSV`
- Use `FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')`

In [ ]:
%sql
-- TODO: Use COPY INTO to load turbine_events.csv into solaris_lab.bronze.turbine_events

### Task 3.3 — Verify idempotency

Count the rows in `turbine_events` now, then run `COPY INTO` again, and count the rows a second time. You should see the same count both times — demonstrating that `COPY INTO` skips already-loaded files.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How does COPY INTO track which files have already been loaded in Databricks?"*

**Hints:**
- Use `SELECT COUNT(*) FROM solaris_lab.bronze.turbine_events` before and after re-running `COPY INTO`
- Both counts should return the same number

In [ ]:
%sql
-- TODO: Count rows in turbine_events (first check)

In [ ]:
%sql
-- TODO: Run COPY INTO again (same command as Task 3.2)

In [ ]:
%sql
-- TODO: Count rows again — confirm the count is unchanged

### Task 3.4 — Build a silver summary table with CTAS

The energy analytics team needs a daily production summary to track output from each site. Use `CREATE TABLE AS SELECT` (`CTAS`) to create `solaris_lab.silver.daily_solar_production` by aggregating from the bronze `solar_readings` table.

The summary should include:
- `reading_date` — the date portion of `reading_timestamp`
- `site_id`
- `total_power_kw` — sum of `power_kw` for Normal-status readings only
- `avg_irradiance` — average `irradiance_w_m2`
- `fault_count` — count of readings where `status = 'Fault'`

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I use CREATE TABLE AS SELECT in Databricks SQL to aggregate data from an existing table, grouping by date and site?"*

**Hints:**
- Use `CREATE OR REPLACE TABLE solaris_lab.silver.daily_solar_production AS SELECT ...`
- Use `CAST(reading_timestamp AS DATE)` or `DATE(reading_timestamp)` to extract the date
- Use `SUM(CASE WHEN status = 'Normal' THEN power_kw ELSE 0 END)` for conditional aggregation
- Group by `reading_date` and `site_id`

In [ ]:
%sql
-- TODO: Create or replace solaris_lab.silver.daily_solar_production
-- Aggregate from solaris_lab.bronze.solar_readings
-- Include: reading_date, site_id, total_power_kw, avg_irradiance, fault_count

In [ ]:
%sql
-- TODO: Query the summary table to verify the results
-- Order by reading_date and site_id

## Exercise 4: Continuous File Ingestion with Auto Loader

New solar readings arrive every 15 minutes from the SCADA system and land as individual CSV files in a cloud storage directory. Rather than scheduling batch jobs to scan for new files, you will use **Auto Loader** to automatically detect and ingest them — processing each file exactly once.

Auto Loader uses the `cloudFiles` source format with `spark.readStream`. The `trigger(availableNow=True)` option processes all currently available files and then stops, which is ideal for scheduled or on-demand ingestion.

### ✅ Provided: Simulate incoming data files

Run the cell below. It creates three separate CSV files in an `incoming/` sub-directory of your volume, simulating three consecutive SCADA export waves. Auto Loader will detect and process all of them.

In [ ]:
# ✅ Run this cell — it writes three incoming data files

import os

incoming_dir = "/Volumes/solaris_lab/bronze/raw_files/incoming"
os.makedirs(incoming_dir, exist_ok=True)

wave1 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R036,SITE_DE_001,P-DE-001,2025-06-03 09:00:00,103.4,558,20.3,Normal
R037,SITE_DE_001,P-DE-002,2025-06-03 09:00:00,99.7,545,20.6,Normal
R038,SITE_DE_001,P-DE-003,2025-06-03 09:00:00,105.1,562,20.1,Normal
R039,SITE_DE_001,P-DE-004,2025-06-03 09:00:00,101.3,552,20.4,Warning
R040,SITE_DE_001,P-DE-005,2025-06-03 09:00:00,97.8,540,20.8,Normal"""

wave2 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R041,SITE_ES_001,P-ES-001,2025-06-03 09:15:00,148.2,765,29.4,Normal
R042,SITE_ES_001,P-ES-002,2025-06-03 09:15:00,143.5,755,29.8,Normal
R043,SITE_ES_001,P-ES-003,2025-06-03 09:15:00,151.9,778,29.1,Normal
R044,SITE_ES_001,P-ES-004,2025-06-03 09:15:00,146.7,761,29.5,Normal
R045,SITE_ES_001,P-ES-005,2025-06-03 09:15:00,140.3,748,30.1,Normal"""

wave3 = """reading_id,site_id,panel_id,reading_timestamp,power_kw,irradiance_w_m2,temperature_c,status
R046,SITE_DE_001,P-DE-001,2025-06-03 09:30:00,108.6,570,20.9,Normal
R047,SITE_DE_001,P-DE-002,2025-06-03 09:30:00,104.9,562,21.2,Normal
R048,SITE_DE_001,P-DE-003,2025-06-03 09:30:00,110.3,575,20.7,Normal
R049,SITE_DE_001,P-DE-004,2025-06-03 09:30:00,0.0,565,21.0,Fault
R050,SITE_DE_001,P-DE-005,2025-06-03 09:30:00,102.7,558,21.4,Normal"""

for name, content in [("wave_1.csv", wave1), ("wave_2.csv", wave2), ("wave_3.csv", wave3)]:
    with open(f"{incoming_dir}/{name}", "w") as f:
        f.write(content)

print(f"✅ Wrote 3 incoming files to {incoming_dir}/")
print("   wave_1.csv — 5 readings from SITE_DE_001 at 09:00")
print("   wave_2.csv — 5 readings from SITE_ES_001 at 09:15")
print("   wave_3.csv — 5 readings from SITE_DE_001 at 09:30")

### Task 4.1 — Configure the Auto Loader read stream

Use `spark.readStream` with the `cloudFiles` format to create a streaming DataFrame that monitors the `incoming/` directory for new CSV files. Auto Loader will automatically track which files it has processed.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I configure Auto Loader in PySpark using spark.readStream and the cloudFiles format to read CSV files from a Unity Catalog volume, with schemaLocation?"*

**Hints:**
- Use `.format("cloudFiles")` on `spark.readStream`
- Set `cloudFiles.format` to `csv`
- Set `cloudFiles.schemaLocation` to a path inside the volume, e.g. `/Volumes/solaris_lab/bronze/raw_files/schema`
- Set `cloudFiles.inferColumnTypes` to `true` so numeric columns are typed correctly
- Load from `/Volumes/solaris_lab/bronze/raw_files/incoming/`

In [ ]:
# TODO: Configure the Auto Loader read stream
# Read from /Volumes/solaris_lab/bronze/raw_files/incoming/
# Use CSV format with header, infer column types, store schema at /Volumes/solaris_lab/bronze/raw_files/schema

df_autoloader = (
    spark.readStream
    # ... add your configuration here
)

### Task 4.2 — Write the stream to a Delta table

Write the Auto Loader stream to `solaris_lab.bronze.solar_readings_stream`. Use `trigger(availableNow=True)` to process all currently available files and then stop — this is the right pattern for scheduled ingestion jobs.

You must also specify a **checkpoint location** so that Auto Loader can track progress and resume correctly if the job restarts.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I write an Auto Loader (spark.readStream cloudFiles) stream to a Unity Catalog table using writeStream, trigger(availableNow=True), and a checkpoint location?"*

**Hints:**
- Use `.writeStream` on your streaming DataFrame
- Set `checkpointLocation` to `/Volumes/solaris_lab/bronze/raw_files/checkpoint`
- Use `.trigger(availableNow=True)` to process all pending files and stop
- Use `.toTable("solaris_lab.bronze.solar_readings_stream")` as the sink
- Call `.awaitTermination()` on the query object to wait for completion

In [ ]:
# TODO: Write the Auto Loader stream to solaris_lab.bronze.solar_readings_stream
# Use trigger(availableNow=True) and set a checkpoint location

query = (
    df_autoloader.writeStream
    # ... add your configuration here
)

query.awaitTermination()

### Task 4.3 — Verify the Auto Loader ingestion

Query `solaris_lab.bronze.solar_readings_stream` to confirm all 15 rows (5 per wave × 3 waves) were loaded. Then run the write stream cell a second time to confirm that no duplicate rows are created — demonstrating Auto Loader's exactly-once guarantee.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How can I verify that Auto Loader ingested the correct number of rows and didn't create duplicates?"*

**Hints:**
- Use `SELECT COUNT(*) FROM solaris_lab.bronze.solar_readings_stream`
- Re-run the cell in Task 4.2 and count again — the result should still be 15

In [ ]:
%sql
-- TODO: Count all rows in solar_readings_stream


-- TODO: Show a sample of rows ordered by reading_timestamp

## ✅ Clean up (optional)

If you want to remove the resources created in this lab, run the cell below.

> ⚠️ This permanently deletes the `solaris_lab` catalog and all its contents, including all tables and volumes.

In [ ]:
%sql
-- Uncomment and run to remove all lab resources
-- DROP CATALOG IF EXISTS solaris_lab CASCADE;